In [6]:
!pip install tqdm

import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from math import log2
from collections import defaultdict

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ======================================================
# 1. 資料準備 (同前)
# ======================================================
ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv")

movies["Genres"] = movies["Genres"].fillna("Unknown").astype(str)

user_ids = sorted(ratings["UserID"].unique())
movie_ids = sorted(ratings["MovieID"].unique())
user2idx = {u: i for i, u in enumerate(user_ids)}
movie2idx = {m: i for i, m in enumerate(movie_ids)}
n_users = len(user_ids)
n_movies = len(movie_ids)

# ======================================================
# 2. 矩陣與切割 (同前)
# ======================================================
print("Building matrices...")
rating_matrix = torch.zeros((n_users, n_movies), dtype=torch.float32).to(device)

for _, row in ratings.iterrows():
    u = user2idx[row["UserID"]]
    m = movie2idx[row["MovieID"]]
    rating_matrix[u, m] = float(row["Rating"])

train_matrix = rating_matrix.clone()
test_entries = []
TEST_RATIO = 0.2

for u in range(n_users):
    rated_items = torch.where(rating_matrix[u] > 0)[0]
    num_rated = len(rated_items)
    if num_rated < 5: continue
    perm = torch.randperm(num_rated)
    rated_items = rated_items[perm]
    n_test = max(1, int(num_rated * TEST_RATIO))
    test_indices = rated_items[:n_test]
    for m_idx in test_indices:
        r = rating_matrix[u, m_idx].item()
        test_entries.append((u, m_idx.item(), r))
        train_matrix[u, m_idx] = 0.0

# ======================================================
# 3. [升級] 計算偏差 (Biases) 與 去中心化
# ======================================================
print("Computing Biases (Pearson logic)...")

# 1. 全域平均 (Global Mean)
global_mean = train_matrix[train_matrix > 0].mean()

# 2. 物品偏差 (Item Bias): 該電影平均比全域平均高多少
# sum(R_ui - global) / count
item_sums = train_matrix.sum(dim=0)
item_counts = (train_matrix > 0).float().sum(dim=0)
# 避免除以0，加上正則化項 (例如 +5) 讓資料少的偏差保守一點
item_biases = (item_sums - item_counts * global_mean) / (item_counts + 5.0)
item_biases[item_counts == 0] = 0.0

# 3. 使用者偏差 (User Bias): 該使用者平均比 (全域+物品) 高多少
# sum(R_ui - global - b_i) / count
train_diff = train_matrix.clone()
mask = (train_matrix > 0)
# 扣除全域與物品偏差
train_diff[mask] -= (global_mean + item_biases.unsqueeze(0).expand_as(train_matrix)[mask])

user_sums = train_diff.sum(dim=1)
user_counts = (train_matrix > 0).float().sum(dim=1)
user_biases = user_sums / (user_counts + 5.0)
user_biases[user_counts == 0] = 0.0

# 4. 建立「殘差矩陣」(Residual Matrix)
# 這是真正的 Pearson 核心：我們只對「意料之外的評分」做相似度計算
# Residual = Rating - (Global + UserBias + ItemBias)
residual_matrix = torch.zeros_like(train_matrix)
residual_matrix[mask] = train_matrix[mask] - (
    global_mean +
    user_biases.unsqueeze(1).expand_as(train_matrix)[mask] +
    item_biases.unsqueeze(0).expand_as(train_matrix)[mask]
)

# ======================================================
# 4. TF-IDF Genre (保留你的優化)
# ======================================================
print("Computing Genre Pref (TF-IDF)...")
all_genres = sorted(set("|".join(movies["Genres"]).split("|")))

# 移除可能的空字串 (如果有的話)
if "" in all_genres: all_genres.remove("")
if "Unknown" in all_genres: all_genres.remove("Unknown")

genre2idx = {g: i for i, g in enumerate(all_genres)}
n_genres = len(all_genres)

movie_genre_matrix = torch.zeros((n_movies, n_genres), device=device)
for _, row in movies.iterrows():
    if row["MovieID"] in movie2idx:
        m_idx = movie2idx[row["MovieID"]]
        genres_str = str(row["Genres"])
        for g in row["Genres"].split("|"):
            if g in genre2idx:
                movie_genre_matrix[m_idx, genre2idx[g]] = 1.0

# TF-IDF
genre_counts = movie_genre_matrix.sum(dim=0)
idf = torch.log(n_movies / (genre_counts + 1.0)).unsqueeze(0)
weighted_movie_genre = movie_genre_matrix * idf

# 計算 User Pref
raw_genre_scores = torch.matmul(train_matrix, weighted_movie_genre)
user_genre_pref = F.normalize(raw_genre_scores, p=1, dim=1)

# Genre Prediction
genre_prediction_score = torch.matmul(user_genre_pref, weighted_movie_genre.T)

# ======================================================
# 5. Item-Based KNN on RESIDUALS
# ======================================================
print("Computing Pearson Similarity (Item-Item on Residuals)...")

# 對殘差做 Cosine Similarity = Pearson Correlation
item_norm = F.normalize(residual_matrix.T, p=2, dim=1)
item_similarity = torch.matmul(item_norm, item_norm.T)
item_similarity.fill_diagonal_(0)

# ======================================================
# 6. 評估與混合
# ======================================================
def evaluate(pred_matrix, test_data, train_mat, top_k=10):
    # Standard Eval Code
    y_true, y_pred = [], []
    for u, m, r in test_data:
        y_true.append(r)
        y_pred.append(pred_matrix[u, m].item())

    y_true = torch.tensor(y_true, device=device)
    y_pred = torch.tensor(y_pred, device=device)

    RMSE = torch.sqrt(torch.mean((y_true - y_pred)**2)).item()
    MAE = torch.mean(torch.abs(y_true - y_pred)).item()

    test_targets = defaultdict(set)
    for u, m, r in test_data:
        if r >= 3.0: test_targets[u].add(m)

    rank_matrix = pred_matrix.clone()
    rank_matrix[train_mat > 0] = -float('inf')

    topk_indices = torch.topk(rank_matrix, top_k, dim=1).indices

    precisions, recalls, ndcgs = [], [], []
    valid_users = [u for u in test_targets.keys()]

    for u in valid_users:
        targets = test_targets[u]
        recs = topk_indices[u].tolist()
        hits = sum(1 for item in recs if item in targets)

        precisions.append(hits / top_k)
        recalls.append(hits / len(targets))

        dcg = 0.0
        for rank, item in enumerate(recs, start=1):
            if item in targets:
                dcg += 1.0 / log2(rank + 1)
        idcg = sum(1.0 / log2(i + 1) for i in range(1, min(top_k, len(targets)) + 1))
        ndcgs.append(dcg / idcg if idcg > 0 else 0)

    return RMSE, MAE, np.mean(precisions), np.mean(recalls), np.mean(ndcgs)


results = []
knn_k_list = [10, 20]
# Genre Weight 可以嘗試低一點，因為 Pearson 很強
genre_weights = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]

# Z-Score Alignment Stats
genre_mean = genre_prediction_score.mean()
genre_std = genre_prediction_score.std()

print("Starting Bias-Corrected KNN Evaluation...")

for K in knn_k_list:
    print(f"\n=== Processing K = {K} ===")

    # 1. 針對目前的 K 計算 KNN 預測矩陣
    # ------------------------------------------------
    topk_val, topk_idx = torch.topk(item_similarity, K, dim=1)
    item_sim_mask = torch.zeros_like(item_similarity)
    item_sim_mask.scatter_(1, topk_idx, topk_val)

    # 預測殘差 (Weighted sum of residuals)
    weighted_sum = torch.matmul(residual_matrix, item_sim_mask)
    weight_count = torch.matmul((residual_matrix != 0).float(), item_sim_mask.abs())
    pred_residual = weighted_sum / (weight_count + 1e-8)

    # 還原分數: Pred = Global + UserBias + ItemBias + Pred_Residual
    base_pred = global_mean + user_biases.unsqueeze(1) + item_biases.unsqueeze(0)
    final_knn_pred = base_pred + pred_residual
    final_knn_pred = torch.clamp(final_knn_pred, 1.0, 5.0)

    # 2. 針對目前的 K 重新計算 Scaling 參數
    # ------------------------------------------------
    knn_mean = final_knn_pred.mean()
    knn_std = final_knn_pred.std()

    # 讓 Genre 分數的分佈跟目前的 KNN 預測一致
    aligned_genre = (genre_prediction_score - genre_mean) / (genre_std + 1e-9) * knn_std + knn_mean

    # 3. 測試不同 Genre Weight
    # ------------------------------------------------
    for gw in genre_weights:
        kw = 1.0 - gw
        hybrid_pred = kw * final_knn_pred + gw * aligned_genre

        rmse, mae, p10, r10, ndcg = evaluate(hybrid_pred, test_entries, train_matrix)
        results.append([K, gw, rmse, mae, p10, r10, ndcg])

        print(f"K={K} | GW={gw:.1f} | RMSE={rmse:.4f} | MAE={mae:.4f} | P@10={p10:.4f} | R@10={r10:.4f} | nDCG={ndcg:.4f}")

df = pd.DataFrame(results, columns=["K", "Genre_W", "RMSE", "MAE", "P@10", "R@10", "nDCG"])
print("\nFinal Results:\n", df)

Using device: cuda
Building matrices...
Computing Biases (Pearson logic)...
Computing Genre Pref (TF-IDF)...
Computing Pearson Similarity (Item-Item on Residuals)...
Starting Bias-Corrected KNN Evaluation...

=== Processing K = 10 ===
K=10 | GW=0.0 | RMSE=0.9492 | MAE=0.7235 | P@10=0.0528 | R@10=0.0302 | nDCG=0.0561
K=10 | GW=0.1 | RMSE=0.9388 | MAE=0.7244 | P@10=0.0930 | R@10=0.0521 | nDCG=0.1108
K=10 | GW=0.2 | RMSE=0.9492 | MAE=0.7397 | P@10=0.1131 | R@10=0.0626 | nDCG=0.1314
K=10 | GW=0.3 | RMSE=0.9798 | MAE=0.7687 | P@10=0.1270 | R@10=0.0679 | nDCG=0.1452
K=10 | GW=0.4 | RMSE=1.0288 | MAE=0.8099 | P@10=0.1308 | R@10=0.0655 | nDCG=0.1486
K=10 | GW=0.5 | RMSE=1.0938 | MAE=0.8620 | P@10=0.1265 | R@10=0.0604 | nDCG=0.1429

=== Processing K = 20 ===
K=20 | GW=0.0 | RMSE=0.9396 | MAE=0.7160 | P@10=0.0342 | R@10=0.0204 | nDCG=0.0361
K=20 | GW=0.1 | RMSE=0.9311 | MAE=0.7180 | P@10=0.0716 | R@10=0.0427 | nDCG=0.0870
K=20 | GW=0.2 | RMSE=0.9450 | MAE=0.7357 | P@10=0.0947 | R@10=0.0542 | nDC